In [11]:
import pandas as pd
import pickle

from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

In [2]:
pre_data = pd.read_csv("preprocessed.csv")

In [3]:
#文字列を含む列のエンコード
categorical_columns = ["Cabin", "Embarked", "Ticket_ID"]
categorical_indices = [pre_data.columns.get_loc(col) for col in categorical_columns]

In [4]:
unique_values_dict = {col: pre_data[col].unique() for col in categorical_columns}

In [5]:
for col in categorical_columns:
    pre_data[col] = pre_data[col].map(lambda x: x if x in unique_values_dict[col] else "other")

column_trans = ColumnTransformer(
    [("onehot", OneHotEncoder(), categorical_indices)], 
    remainder="passthrough"
) 

column_trans.fit(pre_data)
data_encoded = column_trans.transform(pre_data)

In [6]:
#エンコードしたデータの分割及びsurvivedデータの取得
X = data_encoded[:891]
y = pd.read_csv("survived.csv")
test = data_encoded[891:]

In [7]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("test shape:", test.shape)

X shape: (891, 56)
y shape: (891, 1)
test shape: (418, 56)


In [8]:
#最適化及びモデル作成
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

model = RandomForestClassifier()

grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='accuracy')
grid_search.fit(X, y.values.ravel())

best_params = grid_search.best_params_
print("Best Parameters:", best_params)

best_model = grid_search.best_estimator_

Best Parameters: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}


In [9]:
#モデルの保存
with open("Titanic_model.pickle", mode="wb") as fp:
    pickle.dump(best_model, fp)

In [12]:
cv_scores = cross_val_score(grid_search.best_estimator_, X, y.values.ravel(), cv=5, scoring='accuracy')

In [13]:
print("Cross Validation Scores:", cv_scores)

Cross Validation Scores: [0.70391061 0.8258427  0.87078652 0.81460674 0.84269663]


In [14]:
print("Mean Score:", cv_scores.mean())

Mean Score: 0.8115686397589605
